# Aussie Weather on PySpark (Colab)

## Part 1: Install PySpark and create SparkSession

In [ ]:
!pip install pyspark -q

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("AussieWeather")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
print("SparkSession created:", spark.version)

## Upload CSV and set path

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise ValueError("No file uploaded.")
csv_name = list(uploaded.keys())[0]
with open(f"/content/{csv_name}", "wb") as f:
    f.write(uploaded[csv_name])
csv_path = f"/content/{csv_name}"
print(f"Uploaded: {csv_name}")

## Load data with Spark (cluster parallelism)

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)
df.cache()
n = df.count()
cols = df.columns
print(f"Loaded {n} records")
print("Columns:", cols)

## Descriptive statistics (Spark)

In [ ]:
from pyspark.sql import functions as F

numeric_cols = [c for c in df.columns if df.schema[c].dataType.simpleString() in ("double", "int", "long", "float")]
desc = df.select(numeric_cols).describe()
print("=== Descriptive Statistics ===\n")
desc.show(20, truncate=False)

median_row = {}
for c in numeric_cols:
    q = df.approxQuantile(c, [0.5], 0.01)
    if q:
        median_row[c] = round(q[0], 2)
print("MEDIAN (approx):")
for k, v in median_row.items():
    print(f"  {k}: {v}")

## Write summary to file

In [ ]:
summary_path = "/content/weather_summary.txt"
desc_pandas = desc.toPandas()
with open(summary_path, "w") as f:
    f.write("=== Weather Data Summary ===\n\n")
    f.write(f"Total records: {n}\n\n")
    f.write(f"Columns: {cols}\n\n")
    f.write("Statistical Summary (Spark describe):\n")
    f.write(desc_pandas.to_string())
print(f"Summary written to {summary_path}")

## Operations

In [ ]:
rainy = df.filter(F.col("Rainfall") > 0)
print(f"Rainy days: {rainy.count()} records")

hot = df.filter(F.col("MaxTemp") > 35)
print(f"Hot days >35°C: {hot.count()} records")

temps_f = df.select((F.col("MaxTemp") * 9/5 + 32).alias("MaxTemp_F")).limit(3)
sample_f = [r.MaxTemp_F for r in temps_f.collect()]
print(f"Converted temperatures to Fahrenheit; sample: {sample_f[0]:.1f}°F, {sample_f[1]:.1f}°F, {sample_f[2]:.1f}°F...")

total_rain = df.agg(F.sum(F.coalesce(F.col("Rainfall"), F.lit(0)))).collect()[0][0]
print(f"Total rainfall: {total_rain:.2f}mm")

avg_temp = df.agg(F.avg("MaxTemp")).collect()[0][0]
print(f"Average MaxTemp: {avg_temp:.2f}°C")

rainfall_by_loc = df.groupBy("Location").agg(F.avg("Rainfall").alias("avg_rain")).orderBy(F.desc("avg_rain")).limit(5)
print("\n--- Top 5 Locations by Rainfall ---")
for row in rainfall_by_loc.collect():
    print(f"  {row.Location}: {row.avg_rain:.2f}mm")

## Plots 

In [ ]:
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs("/content/plots", exist_ok=True)

In [ ]:
max_temp_pd = df.select("MaxTemp").toPandas()
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=max_temp_pd, x="MaxTemp", kde=True, ax=ax, color="coral")
ax.set_title("Distribution of Maximum Temperatures")
ax.set_xlabel("Maximum Temperature (°C)")
plt.tight_layout()
plt.savefig("/content/plots/temp_distribution.png")
plt.show()
print("Saved temp_distribution.png")

In [ ]:
rainfall_loc_pd = (
    df.groupBy("Location")
    .agg(F.avg("Rainfall").alias("avg_rain"))
    .orderBy(F.desc("avg_rain"))
    .limit(10)
    .toPandas()
)
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=rainfall_loc_pd["avg_rain"], y=rainfall_loc_pd["Location"], ax=ax, hue=rainfall_loc_pd["Location"], legend=False, palette="Blues_d")
ax.set_title("Top 10 Locations by Average Rainfall")
ax.set_xlabel("Average Rainfall (mm)")
plt.tight_layout()
plt.savefig("/content/plots/rainfall_by_location.png")
plt.show()
print("Saved rainfall_by_location.png")

In [ ]:
trends_pd = df.select("MinTemp", "MaxTemp").limit(1000).toPandas()
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(range(len(trends_pd)), trends_pd["MaxTemp"], label="Max Temp", color="red", alpha=0.7)
ax.plot(range(len(trends_pd)), trends_pd["MinTemp"], label="Min Temp", color="blue", alpha=0.7)
ax.fill_between(range(len(trends_pd)), trends_pd["MinTemp"], trends_pd["MaxTemp"], alpha=0.2)
ax.set_title("Temperature Trends (Min and Max)")
ax.set_xlabel("Record Index")
ax.set_ylabel("Temperature (°C)")
ax.legend()
plt.tight_layout()
plt.savefig("/content/plots/temp_trends.png")
plt.show()
print("Saved temp_trends.png")

In [ ]:
numeric_pd = df.select(numeric_cols).toPandas()
corr = numeric_pd.corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f", ax=ax, square=True)
ax.set_title("Correlation Heatmap of Weather Variables")
plt.tight_layout()
plt.savefig("/content/plots/correlation_heatmap.png")
plt.show()
print("Saved correlation_heatmap.png")

In [ ]:
rainy_count = df.filter(F.col("Rainfall") > 0).count()
dry_count = df.filter((F.col("Rainfall") == 0) | F.col("Rainfall").isNull()).count()
fig, ax = plt.subplots(figsize=(8, 8))
ax.pie([rainy_count, dry_count], labels=["Rainy Days", "Dry Days"], autopct="%1.1f%%",
       colors=["steelblue", "lightyellow"], explode=(0.05, 0))
ax.set_title("Proportion of Rainy vs Dry Days")
plt.tight_layout()
plt.savefig("/content/plots/rainy_vs_dry.png")
plt.show()
print("Saved rainy_vs_dry.png")

## Done

Summary and plots are under `/content/`.